# Exploratory Data Analysis: Abalone Dataset

The objective of this analysis is to explore the **Abalone dataset** to understand the physical characteristics of abalones and how they relate to the target variable: **`Rings`**.

### Problem Context
- In biology, the age of an abalone is determined by cutting the shell through the cone, staining it, and counting the number of growth rings under a microscope (estimated age in years = `Rings + 1.5`).
- Counting rings is destructive, labor-intensive, and time-consuming.
- The machine learning goal is to accurately predict the number of rings using non-destructive, easily measurable physical dimensions and weights.

## 1. Imports and Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

## 2. Data Loading & Initial Inspection

In [ ]:
data_path = os.path.join("..", "data", "abalone.csv")
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(10)

In [ ]:
df.info()

## 3. Data Integrity & Anomaly Checks

In [ ]:
# Check missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

In [ ]:
# Summary statistics for numerical features
df.describe().T

### Anomaly Investigation: Zero and Extreme Heights
Let's check physical measurements that might represent data entry errors or extreme outliers, such as `Height == 0`.

In [ ]:
# Inspect rows where Height is zero
zero_heights = df[df["Height"] == 0]
print(f"Number of records with Height == 0: {len(zero_heights)}")
zero_heights

In [ ]:
# Inspect upper extremes of Height
df.sort_values(by="Height", ascending=False)[["Sex", "Length", "Diameter", "Height", "Whole weight", "Rings"]].head(5)

**Key Observations**:
- Two records have `Height == 0`, which is physically impossible for living abalones with recorded weights > 0.2g.
- Two records have unusually high height values (`Height = 1.13` and `0.515`), while the 75th percentile is only `0.165`. These are clear anomalies/outliers that should be treated during preprocessing.

## 4. Target Variable Analysis (`Rings`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with KDE
sns.histplot(df["Rings"], kde=True, bins=30, ax=axes[0], color="#2b5c8f")
axes[0].axvline(df["Rings"].median(), color="red", linestyle="--", label=f"Median: {df['Rings'].median():.0f}")
axes[0].axvline(df["Rings"].mean(), color="orange", linestyle="--", label=f"Mean: {df['Rings'].mean():.2f}")
axes[0].set_title("Distribution of Rings (Target Variable)")
axes[0].set_xlabel("Number of Rings")
axes[0].set_ylabel("Count")
axes[0].legend()

# Boxplot
sns.boxplot(x=df["Rings"], ax=axes[1], color="#79addc")
axes[1].set_title("Boxplot of Rings")
axes[1].set_xlabel("Number of Rings")

plt.tight_layout()
plt.show()

print(f"Rings Min: {df['Rings'].min()}, Max: {df['Rings'].max()}")
print(f"Rings Median: {df['Rings'].median()}, Mean: {df['Rings'].mean():.2f}, Skewness: {df['Rings'].skew():.2f}")

**Key Observations**:
- The target variable `Rings` is moderately right-skewed (skewness ≈ 1.11).
- The majority of abalones have between 7 and 12 rings (median = 9, mean = 9.93).
- Older abalones (Rings > 15) are relatively rare, with maximum rings reaching 29.

## 5. Categorical Feature Analysis (`Sex`)
The dataset includes three categories for `Sex`:
- `M`: Male
- `F`: Female
- `I`: Infant

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sex distribution countplot
sns.countplot(data=df, x="Sex", ax=axes[0], palette="Set2")
axes[0].set_title("Count of Abalones by Sex")
axes[0].set_xlabel("Sex")
axes[0].set_ylabel("Count")
for p in axes[0].patches:
    axes[0].annotate(
        f"{int(p.get_height())}",
        (p.get_x() + p.get_width() / 2.0, p.get_height() / 2),
        ha="center",
        va="center",
        color="white",
        fontweight="bold",
    )

# Rings distribution by Sex
sns.boxplot(data=df, x="Sex", y="Rings", ax=axes[1], palette="Set2")
axes[1].set_title("Rings Distribution by Sex")
axes[1].set_xlabel("Sex")
axes[1].set_ylabel("Rings")

plt.tight_layout()
plt.show()

# Summary statistics grouped by Sex
df.groupby("Sex")["Rings"].agg(["count", "mean", "std", "median", "min", "max"])

**Key Observations**:
- The classes are well-balanced (`M`: 1,528, `I`: 1,342, `F`: 1,307).
- Infant abalones (`I`) have a significantly lower ring count (mean ≈ 7.89 rings), while adult males (`M`, mean ≈ 10.71) and females (`F`, mean ≈ 11.13) show very similar distributions.
- Encoding `Sex` (via One-Hot Encoding) is critical to distinguish infants from mature abalones.

## 6. Numerical Features & Correlations
Let's evaluate the linear correlations among all numeric features and with the target `Rings`.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, square=True, linewidths=0.5)
plt.title("Correlation Matrix of Numerical Features and Target", fontsize=14, pad=12)
plt.show()

# Rank correlation with Rings
corr_with_target = corr_matrix["Rings"].sort_values(ascending=False)
print("Correlation with Rings (sorted):")
print(corr_with_target)

**Key Observations**:
- All continuous physical features are strongly positively correlated with each other (many > 0.85), indicating significant **multicollinearity**.
- The features with highest linear correlation to `Rings` are:
  1. `Shell weight` (r = 0.63)
  2. `Diameter` (r = 0.57)
  3. `Height` (r = 0.56)
  4. `Length` (r = 0.56)
- Interestingly, `Shell weight` correlates much more strongly with age than `Shucked weight` (meat weight, r = 0.42), as shells continue thickening and accumulating minerals over time even when flesh growth slows.

## 7. Bivariate Relationships & Growth Saturation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Shell weight vs Rings
sns.scatterplot(data=df, x="Shell weight", y="Rings", hue="Sex", alpha=0.5, ax=axes[0], palette="Set2")
axes[0].set_title("Shell Weight vs Rings")
axes[0].set_xlabel("Shell weight (grams)")
axes[0].set_ylabel("Rings")

# Diameter vs Rings
sns.scatterplot(data=df, x="Diameter", y="Rings", hue="Sex", alpha=0.5, ax=axes[1], palette="Set2")
axes[1].set_title("Diameter vs Rings")
axes[1].set_xlabel("Diameter (mm)")
axes[1].set_ylabel("Rings")

# Whole weight vs Rings
sns.scatterplot(data=df, x="Whole weight", y="Rings", hue="Sex", alpha=0.5, ax=axes[2], palette="Set2")
axes[2].set_title("Whole Weight vs Rings")
axes[2].set_xlabel("Whole weight (grams)")
axes[2].set_ylabel("Rings")

plt.tight_layout()
plt.show()

**Key Observations**:
- There is a noticeable non-linear / plateau relationship: for young abalones (`Rings < 10`), dimensions and weights grow steeply with age.
- For older abalones (`Rings > 12`), size and weight plateau while ring variance expands significantly.
- This implies a simple linear model will perform well on average/younger abalones, but will tend to underestimate the age of older abalones.

## 8. Weight Breakdown & Multicollinearity
Let's verify the physical relationship between `Whole weight` and its components (`Shucked weight` + `Viscera weight` + `Shell weight`).

In [ ]:
df_weight_check = df.copy()
df_weight_check["Sum of parts weight"] = (
    df_weight_check["Shucked weight"] + df_weight_check["Viscera weight"] + df_weight_check["Shell weight"]
)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_weight_check, x="Sum of parts weight", y="Whole weight", alpha=0.4, color="#1f77b4")
plt.plot([0, 1.6], [0, 1.6], color="red", linestyle="--", label="y = x (Equivalence line)")
plt.title("Whole Weight vs Sum of Component Weights")
plt.xlabel("Sum of Parts (Shucked + Viscera + Shell weight)")
plt.ylabel("Whole Weight")
plt.legend()
plt.show()

## 9. Conclusions and Modelling Takeaways

From this exploratory analysis, we establish the following actionable conclusions for building the predictive model:

1. **Missing Data**: No missing values (`NaN`) exist in the dataset.
2. **Data Cleaning**: Two records contain impossible zero height values (`Height == 0`), and extreme outliers exist (`Height > 0.5`). In the modeling pipeline, filtering these anomalous records prevents distortion of linear coefficients.
3. **Categorical Encoding**: `Sex` is a categorical feature with 3 distinct levels (`M`, `F`, `I`). Infants are strongly separated from adults, so `OneHotEncoder` should be applied.
4. **Feature Scaling**: Due to high collinearity and different scales between dimensions (lengths ~ 0.05 - 0.8) and weights (up to 2.8), feature standardization (`StandardScaler`) is necessary for regularized regression (e.g. Ridge).
5. **Expected Model Limitations**: Physical growth plateaus after 10-12 rings, causing variance in ring counts to increase for older abalones. Non-linear models (e.g., Random Forest) will likely capture this transition better than a pure linear model.